<a href="https://colab.research.google.com/github/divyadharshiniug/24ADI006_24BAD023/blob/main/DL_SelfLearning_Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL Self-Learning Assignment 1
TF-IDF, Word2Vec (CBOW/Skip-gram), FastText, Keras Embedding, BERT, visualization.

In [10]:
!pip install gensim scikit-learn tensorflow transformers torch

In [11]:

import os, re, json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from gensim.test.utils import datapath
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec, FastText
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
plt.rcParams.update({'font.family':'DejaVu Sans','axes.edgecolor':'black'})
R={}
docs=[l.strip() for l in open(datapath('lee_background.cor')) if l.strip()]
toks=[simple_preprocess(d,min_len=2) for d in docs]
R['n_docs']=len(docs); R['n_tokens']=sum(map(len,toks))
# --- Traditional
cv=CountVectorizer(stop_words='english',min_df=2); X=cv.fit_transform(docs)
tf=TfidfVectorizer(stop_words='english',min_df=2); T=tf.fit_transform(docs)
R['bow_shape']=X.shape; R['tfidf_shape']=T.shape
R['sparsity']=1-T.nnz/(T.shape[0]*T.shape[1])
vocab=tf.get_feature_names_out()
def tfidf_sim(w,k=5):
    if w not in tf.vocabulary_: return None
    M=T.T.tocsr(); i=tf.vocabulary_[w]
    s=cosine_similarity(M[i],M).ravel(); o=np.argsort(-s)[1:k+1]
    return [(vocab[j],round(float(s[j]),3)) for j in o]
# --- LSA
svd=TruncatedSVD(100,random_state=1); L=svd.fit_transform(T.T)
Ln=L/ (np.linalg.norm(L,axis=1,keepdims=True)+1e-9)
def lsa_sim(w,k=5):
    i=tf.vocabulary_[w]; s=Ln@Ln[i]; o=np.argsort(-s)[1:k+1]
    return [(vocab[j],round(float(s[j]),3)) for j in o]
# --- Word2Vec skipgram & CBOW
w2v=Word2Vec(toks,vector_size=100,window=5,min_count=5,sg=1,epochs=60,seed=1,workers=1)
cb=Word2Vec(toks,vector_size=100,window=5,min_count=5,sg=0,epochs=60,seed=1,workers=1)
ft=FastText(toks,vector_size=100,window=5,min_count=5,sg=1,epochs=40,seed=1,workers=1)
R['w2v_vocab']=len(w2v.wv)
cands=['government','police','israel','australia','president','war','minister','attack']
qs=[q for q in cands if q in w2v.wv and q in tf.vocabulary_][:6]
R['queries']=qs
R['tfidf']={q:tfidf_sim(q) for q in qs}
R['lsa']={q:lsa_sim(q) for q in qs}
R['w2v']={q:[(a,round(b,3)) for a,b in w2v.wv.most_similar(q,topn=5)] for q in qs}
R['cbow']={q:[(a,round(b,3)) for a,b in cb.wv.most_similar(q,topn=5)] for q in qs}
R['ft']={q:[(a,round(b,3)) for a,b in ft.wv.most_similar(q,topn=5)] for q in qs}
# OOV
oov=['governmentally','policemen','israeli-led','australians']
R['oov']={}
for w in oov:
    r={'in_w2v':w in w2v.wv,'in_ft_vocab':w in ft.wv.key_to_index}
    v=ft.wv[w]; r['ft_vec_norm']=round(float(np.linalg.norm(v)),3)
    r['ft_top']=[(a,round(b,3)) for a,b in ft.wv.similar_by_vector(v,topn=4)]
    R['oov'][w]=r
# --- Keras embedding: classification of doc-level synthetic topic? use next-word-free: train sentiment-free model: predict whether doc mentions 'israel' region? Use simple: label docs by keyword group
import tensorflow as tf_
from tensorflow.keras import layers, models
kw=set(['israel','palestinian','arafat','gaza','middle','peace'])
y=np.array([1 if kw & set(t) else 0 for t in toks])
# remove label words from inputs to avoid trivial leakage
toks_in=[[w for w in t if w not in kw] for t in toks]
vec=layers.TextVectorization(max_tokens=5000,output_sequence_length=120)
vec.adapt(tf_.constant([' '.join(t) for t in toks_in]))
Xk=vec(tf_.constant([' '.join(t) for t in toks_in]))
tf_.random.set_seed(1); np.random.seed(1)
id=np.random.permutation(len(y)); ntr=int(.8*len(y)); tr,te=id[:ntr],id[ntr:]
m=models.Sequential([layers.Embedding(5000,50,mask_zero=True),layers.GlobalAveragePooling1D(),layers.Dense(16,activation='relu'),layers.Dense(1,activation='sigmoid')])
m.compile('adam','binary_crossentropy',metrics=['accuracy'])
h=m.fit(Xk.numpy()[tr],y[tr],epochs=30,batch_size=16,validation_data=(Xk.numpy()[te],y[te]),verbose=0)
R['keras']={'pos_rate':float(y.mean()),'train_acc':round(h.history['accuracy'][-1],3),'val_acc':round(h.history['val_accuracy'][-1],3),'emb_shape':list(m.layers[0].get_weights()[0].shape),'params':int(m.count_params())}
kv=m.layers[0].get_weights()[0]; kvoc=vec.get_vocabulary()
# --- Visualisation
words=[w for w in w2v.wv.index_to_key[:150]]
E=np.array([w2v.wv[w] for w in words])
P=PCA(2,random_state=1).fit_transform(E)
Tn=TSNE(2,perplexity=15,random_state=1,init='pca',max_iter=1500).fit_transform(E)
for name,Z in [('pca',P),('tsne',Tn)]:
    fig,ax=plt.subplots(figsize=(7.5,6))
    ax.scatter(Z[:,0],Z[:,1],s=14,c='black')
    for (x,yv),w in zip(Z,words): ax.annotate(w,(x,yv),fontsize=6.5,xytext=(2,2),textcoords='offset points')
    ax.set_title(f'Word2Vec (Skip-gram) embeddings – {name.upper()}'); ax.set_xticks([]);ax.set_yticks([])
    fig.tight_layout(); fig.savefig(f'{name}.png',dpi=170,facecolor='white'); plt.close(fig)
# cluster zoom for query neighborhoods
fig,ax=plt.subplots(figsize=(7.5,5.5))
groups={'government':'o','police':'s','israel':'^'}
ws=[];mk=[]
for q,m_ in groups.items():
    if q in w2v.wv:
        for w in [q]+[a for a,_ in w2v.wv.most_similar(q,topn=10)]: ws.append(w); mk.append(m_)
Ew=np.array([w2v.wv[w] for w in ws]); Zw=PCA(2,random_state=1).fit_transform(Ew)
for (x,yv),w,m_ in zip(Zw,ws,mk):
    ax.scatter(x,yv,marker=m_,c='white',edgecolors='black',s=45); ax.annotate(w,(x,yv),fontsize=7,xytext=(3,3),textcoords='offset points')
ax.set_title('Nearest-neighbour clusters (o: government, □: police, △: israel)'); ax.set_xticks([]);ax.set_yticks([])
fig.tight_layout(); fig.savefig('clusters.png',dpi=170,facecolor='white'); plt.close(fig)
# export projector files
os.makedirs('projector',exist_ok=True)
with open('projector/vectors.tsv','w') as f, open('projector/metadata.tsv','w') as g:
    for w in w2v.wv.index_to_key:
        f.write('\t'.join(f'{x:.5f}' for x in w2v.wv[w])+'\n'); g.write(w+'\n')
R['proj_n']=len(w2v.wv)
# analogy-ish
try: R['analogy']=[(a,round(b,3)) for a,b in w2v.wv.most_similar(positive=['australia','government'],negative=[],topn=3)]
except: pass
json.dump(R,open('results.json','w'),indent=1,default=str)
print(json.dumps(R,indent=1,default=str))

{
 "n_docs": 300,
 "n_tokens": 58152,
 "bow_shape": [
  300,
  3382
 ],
 "tfidf_shape": [
  300,
  3382
 ],
 "sparsity": 0.9790814113936527,
 "w2v_vocab": 1750,
 "queries": [
  "government",
  "police",
  "israel",
  "australia",
  "president",
  "war"
 ],
 "tfidf": {
  "government": [
   [
    "unity",
    0.412
   ],
   [
    "minister",
    0.404
   ],
   [
    "says",
    0.402
   ],
   [
    "la",
    0.398
   ],
   [
    "rua",
    0.398
   ]
  ],
  "police": [
   [
    "alice",
    0.475
   ],
   [
    "springs",
    0.475
   ],
   [
    "vehicle",
    0.435
   ],
   [
    "arrested",
    0.381
   ],
   [
    "kilometres",
    0.378
   ]
  ],
  "israel": [
   [
    "yasser",
    0.732
   ],
   [
    "hamas",
    0.66
   ],
   [
    "palestinian",
    0.601
   ],
   [
    "fatah",
    0.598
   ],
   [
    "arafat",
    0.594
   ]
  ],
  "australia": [
   [
    "says",
    0.428
   ],
   [
    "planning",
    0.343
   ],
   [
    "man",
    0.338
   ],
   [
    "fatality",
    0.3

In [ ]:
!pip install transformers torch

## Contextual embeddings (BERT)

In [12]:
# --- Contextual embeddings with BERT (needs internet; run in Colab) ---
from transformers import AutoTokenizer, AutoModel
import torch
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").eval()
def word_vec(sentence, word):
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        h = bert(**enc).last_hidden_state[0]
    wid = tok.convert_tokens_to_ids(word)
    pos = (enc["input_ids"][0] == wid).nonzero()[0].item()
    return h[pos].numpy()
s1 = "I deposited cash at the river bank."
s2 = "We had a picnic on the bank of the river."
s3 = "The bank approved my home loan."
from sklearn.metrics.pairwise import cosine_similarity as cs
v1, v2, v3 = (word_vec(s1,"bank"), word_vec(s2,"bank"), word_vec(s3,"bank"))
print("river-bank vs money-bank:", cs([v2],[v3])[0,0])
print("money-bank vs money-bank:", cs([v1],[v3])[0,0])
# Static Word2Vec gives ONE vector for "bank"; BERT gives a different vector per context.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


river-bank vs money-bank: 0.4253573
money-bank vs money-bank: 0.59516734
